# Cross-participant MEG Faces Decoding with `coco-pipe`

Using the **Wakeman–Henson multimodal face-processing dataset**, this tutorial asks one focused question: **can famous and scrambled faces be decoded in a participant who was not used to train the classifier?**

`coco-pipe` owns leave-one-subject-out cross-validation, fold-local scaling and alignment, sliding temporal estimation, scoring, aggregation, and plotting. The notebook keeps the complete scientific argument visible: target definition, split audit, alignment scope, group time course, held-out-participant variability, and interpretation.

<div class="alert alert-secondary">
<b>Representation comparison.</b> Sensors and aligned PCA use identical trials, labels, participant folds, classifier, scaling, metric, and time axis. Only the fold-local temporal alignment differs.
</div>

## 0 — Setup and analysis contract

All six prepared participants are used. The visible settings fix the PCA dimension, parallelism, random seed, derivatives location, and optional export directory. Data preparation remains outside this notebook.

<div class="alert alert-info">
<b>Generalization target.</b> A participant is held out in full. Trials from one participant are never divided between training and test data.
</div>

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from coco_pipe.decoding import (
    CVConfig, Experiment, ExperimentConfig, TemporalAlignmentConfig,
    TemporalDecoderConfig,
)
from coco_pipe.decoding.configs import ClassicalModelConfig
from coco_pipe.viz.interactive.decoding import plot_temporal_score_curve
from pca_neural_trajectories.wakeman_henson import (
    MEG_SENSOR_SETS,
    _load_wakeman_henson_container,
)

SEED = 42
N_COMPONENTS = 30
N_JOBS = 1
SUBJECTS = ("01", "02", "03", "04", "05", "06")
SENSOR_SET = "all_sensors"
DERIVATIVES_ROOT = Path.home() / "mne_data" / "ds000117" / "derivatives" / "pca_trajectories"
OUTPUT_ROOT = Path("outputs/tutorial_megfaces_decoding") / SENSOR_SET
if SENSOR_SET not in MEG_SENSOR_SETS:
    raise ValueError(f"SENSOR_SET must be one of {tuple(MEG_SENSOR_SETS)}.")
SAVE_RESULTS = False

## 1 — Define the predictive question

The target is image category: <code>0 = Famous</code> and <code>1 = Scrambled</code>. A trial is an observation, but the participant is the cross-validation and inferential unit. LOSO therefore asks whether category information generalizes to a participant whose labeled trials were not used for training.

## 2 — Load Famous and Scrambled trials

The prepared tensor is loaded as <code>trial × sensor × time</code>. <code>SENSOR_SET</code> optionally restricts the analysis to the official VectorView occipital, temporal, or combined helmet selections; these are sensor positions, not source-localized cortical ROIs. Each participant is whitened within the selected sensors using their empty-room covariance. No local time binning or global standardization is applied before <code>coco-pipe</code>.

Trial counts are documented rather than altered. Balanced accuracy and training-fold class weights handle modest imbalance without resampling test data.

In [ ]:
if not DERIVATIVES_ROOT.exists():
    raise FileNotFoundError(f"Prepared MEG derivatives were not found at {DERIVATIVES_ROOT}.")

container = _load_wakeman_henson_container(
    DERIVATIVES_ROOT, subjects=SUBJECTS, conditions=(1, 3), sensor_set=SENSOR_SET
)
X = np.asarray(container.X, dtype=np.float32)
times = np.asarray(container.coords["time"], dtype=float)
condition = np.asarray(container.y, dtype=int)
subjects = np.asarray(container.coords["subject"]).astype(str)
y = np.where(condition == 1, 0, 1)
trial_ids = (
    np.asarray(container.ids).astype(str)
    if container.ids is not None
    else np.asarray([f"trial-{index}" for index in range(len(X))])
)

counts = pd.crosstab(subjects, np.where(y == 0, "Famous", "Scrambled"))
display(counts)
print(f"{X.shape[0]} trials × {X.shape[1]} sensors × {X.shape[2]} time points")
print(f"Sensor set: {SENSOR_SET}")
print(f"Whitening: {container.meta.get('whitening')}")
del container

## 3 — Configure the target, classifier, and LOSO folds

At each latency, logistic regression is trained on all participants except one and evaluated on the held-out participant. Scaling and balanced class weights are fitted inside the training fold. <code>wrapper="sliding"</code> repeats this model independently across the supplied time axis.

## 4 — Define the aligned comparison

Alignment is fitted independently inside each LOSO fold by <code>coco-pipe</code>. The shared PCA and temporal template use only training participants. The unseen participant's PCA and rotation use that participant's unlabeled trials, so adaptation remains explicitly **transductive**.

<div class="alert alert-warning">
<b>Calibration boundary.</b> Aligned decoding assumes an unlabeled calibration batch from the new participant. It is not zero-calibration inductive decoding.
</div>

In [ ]:
decoder = TemporalDecoderConfig(
    wrapper="sliding",
    base=ClassicalModelConfig(
        estimator="LogisticRegression",
        params={"class_weight": "balanced", "max_iter": 2000},
    ),
    n_jobs=1,
    verbose=False,
)
sensor_config = ExperimentConfig(
    task="classification",
    models={"Logistic regression": decoder},
    metrics=["balanced_accuracy"],
    cv=CVConfig(strategy="leave_one_group_out", shuffle=False),
    use_scaler=True,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=False,
)
aligned_config = sensor_config.model_copy(deep=True)
aligned_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=N_COMPONENTS, adaptation="transductive"
)
experiments = {"Sensors": sensor_config, "Aligned PCA": aligned_config}

## 5 — Inspect the alignment configuration

The only difference between the experiments is the fold-level alignment. Scaling, temporal logistic regression, LOSO splitting, scoring, and the scientific time axis remain identical. Inspecting the resolved configuration makes the number of components and transductive adaptation explicit before fitting.

In [ ]:
aligned_config.temporal_alignment

## 6 — Run LOSO decoding

Both experiments receive the same MEG epochs, labels, participant groups, trial identifiers, and time axis. Alignment is constructed separately inside each fold. We retain temporal summaries, fold-level scores, fit diagnostics, and the exact split definitions.

## 7 — Audit participant separation

A valid outer fold contains exactly one held-out participant and no participant overlap between training and test trials. The audit below is derived from the fitted sensor experiment rather than reconstructed from assumptions.

In [ ]:
results = {}
score_frames = []
fold_frames = []
diagnostic_frames = []
for name, config in experiments.items():
    result = Experiment(config).run(
        X,
        y,
        groups=subjects,
        sample_ids=trial_ids,
        observation_level="epoch",
        inferential_unit="subject",
        time_axis=times,
    )
    results[name] = result
    frame = result.get_temporal_score_summary()
    frame["Model"] = name
    score_frames.append(frame)

    folds = result.get_detailed_scores()
    folds["Representation"] = name
    fold_frames.append(folds)

    diagnostics = result.get_fit_diagnostics()
    diagnostics["Representation"] = name
    diagnostic_frames.append(diagnostics)

temporal_scores = pd.concat(score_frames, ignore_index=True)
fold_scores = pd.concat(fold_frames, ignore_index=True)
fit_diagnostics = pd.concat(diagnostic_frames, ignore_index=True)
display(temporal_scores.head())

split_rows = results["Sensors"].get_splits()
audit_rows = []
for fold in sorted(split_rows["Fold"].unique()):
    fold_rows = split_rows[split_rows["Fold"] == fold]
    train = fold_rows[fold_rows["Set"] == "train"]
    test = fold_rows[fold_rows["Set"] == "test"]
    train_subjects = sorted(train["Group"].astype(str).unique())
    test_subjects = sorted(test["Group"].astype(str).unique())
    overlap = sorted(set(train_subjects) & set(test_subjects))
    audit_rows.append({
        "Fold": int(fold),
        "held_out_subject": ", ".join(test_subjects),
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_train_trials": len(train),
        "n_test_trials": len(test),
        "subject_overlap": ", ".join(overlap),
        "leakage_free": len(overlap) == 0 and len(test_subjects) == 1,
    })

split_audit = pd.DataFrame(audit_rows)
if not split_audit["leakage_free"].all():
    raise RuntimeError("The LOSO audit found participant overlap.")
display(split_audit)

## 8 — Inspect time-resolved generalization

The curve is mean balanced accuracy across held-out participants; the ribbon is the fold-level standard deviation returned by <code>coco-pipe</code>. Binary chance is 0.5 and time zero is image onset. Peak values summarize the curve but are not corrected significance tests across time.

## 9 — Retain held-out-participant variability

A group mean can hide participants with different temporal profiles. Fold heatmaps preserve one row per held-out participant. Two post-onset summaries are then computed per fold: mean balanced accuracy and AUC above chance from 0 to 0.8 s. Aligned-minus-sensor differences remain paired by participant.

<div class="alert alert-secondary">
<b>Interpretation.</b> Broad above-chance periods and consistent held-out-participant behavior are more informative than a single maximum of the group curve. Paired differences are descriptive unless an inferential procedure is declared.
</div>

In [ ]:
colors = {
    "Sensors": "#1b9e77",
    "Aligned PCA": "#2a78d6",
}
figure = plot_temporal_score_curve(
    temporal_scores,
    metric="balanced_accuracy",
    title="Famous versus scrambled faces: LOSO decoding",
    colors=colors,
)
figure.add_hline(y=0.5, line_dash="dot", line_color="#777777")
figure.add_vline(x=0, line_color="#999999")
figure.update_yaxes(title_text="balanced accuracy")
figure.update_xaxes(title_text="time from image onset (s)")
figure.show()

peak_rows = temporal_scores.loc[
    temporal_scores.groupby("Model")["Mean"].idxmax(),
    ["Model", "Time", "Mean", "Std"],
].sort_values("Mean", ascending=False)
display(peak_rows.rename(columns={"Time": "peak time (s)", "Mean": "peak BA"}).round(3))

metric_scores = fold_scores[
    (fold_scores["Metric"] == "balanced_accuracy")
    & fold_scores["Time"].notna()
]
held_out_by_fold = split_audit.set_index("Fold")["held_out_subject"].to_dict()
fold_summary_rows = []
for (representation, fold), rows in metric_scores.groupby(
    ["Representation", "Fold"]
):
    rows = rows.sort_values("Time")
    active = rows[(rows["Time"] >= 0) & (rows["Time"] <= 0.8)]
    peak_index = rows["Value"].idxmax()
    fold_summary_rows.append({
        "Representation": representation,
        "Fold": int(fold),
        "held_out_subject": held_out_by_fold[int(fold)],
        "peak_time_s": float(rows.loc[peak_index, "Time"]),
        "peak_balanced_accuracy": float(rows.loc[peak_index, "Value"]),
        "postonset_mean_balanced_accuracy": float(active["Value"].mean()),
        "postonset_auc_above_chance": float(np.trapezoid(
            active["Value"] - 0.5, active["Time"]
        )),
    })

fold_summary = pd.DataFrame(fold_summary_rows)
representation_summary = (
    fold_summary.groupby("Representation")
    .agg(
        n_folds=("Fold", "nunique"),
        peak_ba_mean=("peak_balanced_accuracy", "mean"),
        peak_ba_std=("peak_balanced_accuracy", "std"),
        postonset_ba_mean=("postonset_mean_balanced_accuracy", "mean"),
        postonset_ba_std=("postonset_mean_balanced_accuracy", "std"),
        postonset_auc_mean=("postonset_auc_above_chance", "mean"),
        postonset_auc_std=("postonset_auc_above_chance", "std"),
    )
    .reset_index()
)
display(representation_summary.round(3))

paired = fold_summary.pivot(
    index=["Fold", "held_out_subject"],
    columns="Representation",
    values=[
        "peak_balanced_accuracy",
        "postonset_mean_balanced_accuracy",
        "postonset_auc_above_chance",
    ],
)
paired_fold_differences = paired.index.to_frame(index=False)
for metric in (
    "peak_balanced_accuracy",
    "postonset_mean_balanced_accuracy",
    "postonset_auc_above_chance",
):
    paired_fold_differences[f"{metric}_aligned_minus_sensors"] = (
        paired[(metric, "Aligned PCA")].to_numpy()
        - paired[(metric, "Sensors")].to_numpy()
    )
display(paired_fold_differences.round(3))

fold_heatmaps = make_subplots(
    rows=1, cols=2, subplot_titles=("Sensors", "Aligned PCA")
)
for column, representation in enumerate(("Sensors", "Aligned PCA"), start=1):
    rows = metric_scores[metric_scores["Representation"] == representation]
    matrix = rows.pivot(index="Fold", columns="Time", values="Value").sort_index()
    fold_heatmaps.add_trace(
        go.Heatmap(
            z=matrix.to_numpy(),
            x=matrix.columns.to_numpy(float),
            y=[f"sub-{held_out_by_fold[int(fold)]}" for fold in matrix.index],
            zmin=0, zmax=1, colorscale="Viridis",
            showscale=column == 2,
        ),
        row=1, col=column,
    )
fold_heatmaps.update_xaxes(title_text="time from image onset (s)")
fold_heatmaps.update_yaxes(title_text="held-out participant", row=1, col=1)
fold_heatmaps.update_layout(
    title="Balanced accuracy for every held-out participant", height=560
)
fold_heatmaps.show()

fold_summary_figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Post-onset mean balanced accuracy",
        "Post-onset AUC above chance",
    ),
)
for column, metric in enumerate((
    "postonset_mean_balanced_accuracy",
    "postonset_auc_above_chance",
), start=1):
    for representation in ("Sensors", "Aligned PCA"):
        rows = fold_summary[fold_summary["Representation"] == representation]
        fold_summary_figure.add_trace(
            go.Box(
                x=[representation] * len(rows), y=rows[metric],
                name=representation, boxpoints="all", jitter=0.25,
                marker_color=colors[representation], showlegend=column == 1,
            ),
            row=1, col=column,
        )
fold_summary_figure.add_hline(y=0.5, line_dash="dot", row=1, col=1)
fold_summary_figure.add_hline(y=0, line_dash="dot", row=1, col=2)
fold_summary_figure.update_layout(
    title="Held-out-participant post-onset summaries", height=500
)
fold_summary_figure.show()

## 10 — Export and takeaway

The notebook has now followed the full evidence chain: participant-disjoint folds, fold-local scaling and alignment, time-resolved group performance, and held-out-participant variability. LOSO supports a cross-participant claim; the aligned model additionally assumes unlabeled calibration data from the new participant.

Peak balanced accuracy remains descriptive without temporal multiplicity correction. Compare representations through sustained performance and paired held-out-participant behavior rather than one maximum.

<div class="alert alert-success">
<b>Reproducible run.</b> Set <code>SAVE_RESULTS=True</code> for the notebook tables and figures. Use <code>scripts/analysis_megfaces_decoding.py</code> for raw result exports, all tidy tables, figures, arrays, provenance manifests, and the self-contained HTML report.
</div>

In [ ]:
if SAVE_RESULTS:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    tables = {
        "temporal_scores": temporal_scores,
        "fold_scores": fold_scores,
        "fit_diagnostics": fit_diagnostics,
        "split_audit": split_audit,
        "peak_summary": peak_rows,
        "fold_summary": fold_summary,
        "representation_summary": representation_summary,
        "paired_fold_differences": paired_fold_differences,
    }
    for name, table in tables.items():
        table.to_csv(OUTPUT_ROOT / f"{name}.csv", index=False)
    figure.write_html(OUTPUT_ROOT / "temporal_decoding.html", include_plotlyjs="cdn")
    fold_heatmaps.write_html(OUTPUT_ROOT / "fold_heatmaps.html", include_plotlyjs="cdn")
    fold_summary_figure.write_html(
        OUTPUT_ROOT / "fold_summaries.html", include_plotlyjs="cdn"
    )
    for name, result in results.items():
        result.export(
            OUTPUT_ROOT / name.lower().replace(" ", "_"),
            config=experiments[name].model_dump(),
            formats=("csv",),
        )
    print(f"Saved results to {OUTPUT_ROOT.resolve()}")
else:
    print("SAVE_RESULTS=False: no files written.")